### 1. GCN architecture and molecular graph features

Purpose: Define the shielding-aware graph neural network and the molecular graph feature representation used throughout the notebook.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 1. GCN architecture and molecular graph features
# Purpose: Define the shielding-aware graph neural network and the molecular graph feature representation used throughout the notebook.
# -----------------------------------------------------------------------------

import os
import random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ---------- Reproducibility ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

def set_global_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

# ---------- Atom-level feature configuration ----------
feature_config = {
    "atom_type": {"encode": True, "type": "categorical"},
    "atom_mass": {"encode": True, "type": "continuous"},
    "hybridization": {"encode": True, "type": "categorical"},
    "is_aromatic": {"encode": False, "type": "boolean"},
    "formal_charge": {"encode": True, "type": "continuous"},
    "chiral_tag": {"encode": False, "type": "boolean"},
    "num_hydrogens": {"encode": True, "type": "continuous"},
    "degree": {"encode": True, "type": "continuous"},
    "implicit_valence": {"encode": True, "type": "continuous"},
    "explicit_valence": {"encode": True, "type": "continuous"},
    "gasteiger_charge": {"encode": True, "type": "continuous"},
    "logp_contrib": {"encode": True, "type": "continuous"},
    "mr_contrib": {"encode": True, "type": "continuous"},
    "estate": {"encode": True, "type": "continuous"},
    "radius": {"encode": True, "type": "continuous"},
    "electronegativity": {"encode": True, "type": "continuous"},
    "is_fluorine": {"encode": False, "type": "boolean"},
}

# =========================
# GCN model (with optional global shielding feature)
# =========================
class GCN(torch.nn.Module):
    def __init__(
        self,
        num_node_features: int,
        hidden_channels: int = 128,
        num_layers: int = 3,
        predictor_hidden_feats: int = 256,
        dropout: float = 0.2,
        use_residual: bool = True,
        use_global_shielding: bool = True,
        global_feat_dim: int = 1,  # shielding_value
    ):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()

        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))

        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        # Predictor head
        # If global shielding is used, we concatenate it after pooling
        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None

        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        # Graph pooling
        g = global_mean_pool(x, batch)  # [num_graphs, hidden_channels]

        # Optional global feature fusion
        if self.use_global_shielding:
            if not hasattr(data, "shielding"):
                raise AttributeError("Data object missing 'shielding' attribute while use_global_shielding=True")
            shielding = data.shielding.view(-1, 1).to(g.dtype)  # [num_graphs, 1]
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()


### 2. PyG dataset construction with shielding

Purpose: Construct the graph-learning datasets and persistent train/validation/test loaders for the shielding-aware workflow.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 2. PyG dataset construction with shielding
# Purpose: Construct the graph-learning datasets and persistent train/validation/test loaders for the shielding-aware workflow.
# -----------------------------------------------------------------------------

from rdkit import Chem
from rdkit.Chem import AllChem, EState, rdMolDescriptors
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# ---------- Paths ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path(".").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GCN_DIR = PROJECT_ROOT / "results" / "gcn"
GCN_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = GCN_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: set this to your cleaned WITH-shielding CSV from Cell 8
CLEANED_CSV = PROCESSED_DIR / "cleaned_shift_data_with_shield.csv"
if not CLEANED_CSV.exists():
    raise FileNotFoundError(f"Cleaned WITH-shielding CSV not found: {CLEANED_CSV}")

# ---------- Load cleaned data ----------
df = pd.read_csv(CLEANED_CSV)

required_cols = {"SMILES", "shift_value", "shielding_value"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Input CSV missing required columns: {missing}")

# ---------- Split (Train/Val/Test) ----------
X = df[["SMILES", "shielding_value"]].copy()
y = df["shift_value"].astype(float).copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1/9), random_state=SEED
)

train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([X_val.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)
test_df  = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

print_section("GCN Dataset Split Summary (WITH shielding)")
print(f"Total samples : {len(df)}")
print(f"Train samples : {len(train_df)}")
print(f"Val samples   : {len(val_df)}")
print(f"Test samples  : {len(test_df)}")

# ---------- Atom-level descriptor extraction ----------
electronegativities = {
    "H": 2.20, "C": 2.55, "N": 3.04, "O": 3.44, "F": 3.98,
    "P": 2.19, "S": 2.58, "Cl": 3.16, "Si": 1.98, "B": 2.04,
    "Br": 2.96, "I": 4.36, "Na": 0.93, "K": 0.82,
}

def add_gasteiger_charges(mol):
    AllChem.ComputeGasteigerCharges(mol)
    charges = []
    for atom in mol.GetAtoms():
        try:
            charges.append(float(atom.GetProp("_GasteigerCharge")))
        except Exception:
            charges.append(0.0)
    return charges

def calculate_atomic_descriptors(mol):
    mol_h = Chem.AddHs(mol)
    logp_contribs, mr_contribs = zip(*rdMolDescriptors._CalcCrippenContribs(mol_h))
    estate_indices = EState.EStateIndices(mol_h)
    pt = Chem.GetPeriodicTable()
    radii = [pt.GetRvdw(atom.GetAtomicNum()) for atom in mol_h.GetAtoms()]
    return logp_contribs, mr_contribs, estate_indices, radii

def molecule_to_graph(mol):
    if mol is None:
        return None

    gasteiger_charges = add_gasteiger_charges(mol)
    logp_contribs, mr_contribs, estate_indices, radii = calculate_atomic_descriptors(mol)

    nodes = []
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        nodes.append({
            "atom_type": atom.GetSymbol(),
            "atom_mass": atom.GetMass(),
            "hybridization": str(atom.GetHybridization()),
            "is_aromatic": atom.GetIsAromatic(),
            "formal_charge": atom.GetFormalCharge(),
            "chiral_tag": atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
            "num_hydrogens": atom.GetTotalNumHs(),
            "degree": atom.GetDegree(),
            "implicit_valence": atom.GetImplicitValence(),
            "explicit_valence": atom.GetExplicitValence(),
            "gasteiger_charge": gasteiger_charges[idx],
            "logp_contrib": logp_contribs[idx],
            "mr_contrib": mr_contribs[idx],
            "estate": estate_indices[idx],
            "radius": radii[idx],
            "electronegativity": electronegativities.get(atom.GetSymbol(), 0.0),
            "is_fluorine": 1 if atom.GetSymbol() == "F" else 0,
        })

    edges = []
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        edges.append([a, b])
        edges.append([b, a])

    if len(edges) == 0:
        edges = [[0, 0]]

    return nodes, edges

# =========================
# UNK-safe categorical encoders + train-only scalers
# =========================
UNK_TOKEN = "__UNK__"

def fit_categorical_maps(all_train_nodes):
    """
    Build vocabulary maps for categorical features from TRAIN only.
    Unknown categories in val/test will be mapped to UNK.
    """
    cat_maps = {}
    for feat, cfg in feature_config.items():
        if cfg["encode"] and cfg["type"] == "categorical":
            vocab = sorted(set([n[feat] for n in all_train_nodes]))
            if UNK_TOKEN not in vocab:
                vocab = vocab + [UNK_TOKEN]
            cat_maps[feat] = {v: i for i, v in enumerate(vocab)}
    return cat_maps

def fit_scalers(all_train_nodes):
    """
    Fit StandardScaler for continuous features using TRAIN only.
    """
    scalers = {}
    for feat, cfg in feature_config.items():
        if cfg["encode"] and cfg["type"] == "continuous":
            arr = np.array([float(n[feat]) for n in all_train_nodes]).reshape(-1, 1)
            sc = StandardScaler()
            sc.fit(arr)
            scalers[feat] = sc
    return scalers

def transform_nodes(nodes, cat_maps, scalers):
    """
    Transform nodes into a numeric matrix (num_nodes, num_node_features).
    """
    out = []
    for n in nodes:
        row = []
        for feat, cfg in feature_config.items():
            if cfg["type"] == "categorical" and cfg["encode"]:
                mp = cat_maps[feat]
                row.append(float(mp.get(n[feat], mp[UNK_TOKEN])))
            elif cfg["type"] == "continuous" and cfg["encode"]:
                row.append(float(scalers[feat].transform([[float(n[feat])]])[0][0]))
            elif cfg["type"] == "boolean":
                row.append(float(int(n[feat])))
        out.append(row)
    return np.array(out, dtype=np.float32)

def df_to_pyg_list(data_df, cat_maps, scalers):
    data_list = []
    invalid = 0

    for i, row in data_df.iterrows():
        smiles = str(row["SMILES"]).strip()
        y_val = float(row["shift_value"])
        shielding_val = float(row["shielding_value"])

        mol = Chem.MolFromSmiles(smiles)
        g = molecule_to_graph(mol)
        if g is None:
            invalid += 1
            continue

        nodes, edges = g
        x_np = transform_nodes(nodes, cat_maps, scalers)
        x = torch.tensor(x_np, dtype=torch.float)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        y_tensor = torch.tensor([y_val], dtype=torch.float)

        data = Data(x=x, edge_index=edge_index, y=y_tensor, smiles=smiles)
        data.shielding = torch.tensor([shielding_val], dtype=torch.float)
        data_list.append(data)

    return data_list, invalid

# ---------- Collect all TRAIN nodes for fitting encoders/scalers ----------
train_nodes_all = []
invalid_train = 0
for _, row in train_df.iterrows():
    mol = Chem.MolFromSmiles(str(row["SMILES"]).strip())
    g = molecule_to_graph(mol)
    if g is None:
        invalid_train += 1
        continue
    nodes, _ = g
    train_nodes_all.extend(nodes)

cat_maps = fit_categorical_maps(train_nodes_all)
scalers = fit_scalers(train_nodes_all)

# Save encoders/scalers for reproducibility
ENC_SCALER_PATH = GCN_DIR / "encoders_scalers.pkl"
joblib.dump({"cat_maps": cat_maps, "scalers": scalers, "UNK_TOKEN": UNK_TOKEN}, ENC_SCALER_PATH)

# ---------- Build PyG datasets ----------
train_list, inv1 = df_to_pyg_list(train_df, cat_maps, scalers)
val_list,   inv2 = df_to_pyg_list(val_df, cat_maps, scalers)
test_list,  inv3 = df_to_pyg_list(test_df, cat_maps, scalers)

print_section("Graph Construction Summary")
print(f"Invalid SMILES removed (train/val/test): {inv1} / {inv2} / {inv3}")
print(f"Train graphs: {len(train_list)} | Val graphs: {len(val_list)} | Test graphs: {len(test_list)}")

# ---------- DataLoaders ----------
BATCH_SIZE = 64
train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(val_list, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_list, batch_size=BATCH_SIZE, shuffle=False)

torch.save(train_loader, GCN_DIR / "train_loader.pth")
torch.save(valid_loader, GCN_DIR / "valid_loader.pth")
torch.save(test_loader,  GCN_DIR / "test_loader.pth")

train_df.to_csv(TABLE_DIR / "train.csv", index=False)
val_df.to_csv(TABLE_DIR / "valid.csv", index=False)
test_df.to_csv(TABLE_DIR / "test.csv", index=False)

num_node_features = train_list[0].x.shape[1]
print(f"\nNode feature dimension (data.x): {num_node_features}")
print(f"Saved loaders to: {GCN_DIR.resolve()}")


### 3. Optuna search for the shielding-aware GCN

Purpose: Optimize the main GCN hyperparameters with Optuna while retaining the original search strategy.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 3. Optuna search for the shielding-aware GCN
# Purpose: Optimize the main GCN hyperparameters with Optuna while retaining the original search strategy.
# -----------------------------------------------------------------------------

import os
import numpy as np
import optuna
from copy import deepcopy
import torch

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

train_loader = torch.load(GCN_DIR / "train_loader.pth",weights_only=False)
valid_loader = torch.load(GCN_DIR / "valid_loader.pth",weights_only=False)

num_node_features = train_loader.dataset[0].x.shape[1]

class EarlyStopping:
    def __init__(self, patience=50, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def step(self, val_loss: float):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.counter = 0
            return
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def objective(trial):
    hidden_channels = trial.suggest_categorical("hidden_channels", [64, 128, 256, 512])
    num_layers = trial.suggest_int("num_layers", 2, 5)
    predictor_hidden_feats = trial.suggest_categorical("predictor_hidden_feats", [128, 256, 512, 1024])
    dropout = trial.suggest_float("dropout", 0.05, 0.6)
    lr = trial.suggest_float("lr", 1e-4, 5e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    # NOTE: make sure your GCN signature matches these arguments.
    model = GCN(
        num_node_features=num_node_features,
        hidden_channels=hidden_channels,
        num_layers=num_layers,
        predictor_hidden_feats=predictor_hidden_feats,
        dropout=dropout,
        use_residual=True,
        use_global_shielding=True,
        global_feat_dim=1,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.MSELoss()

    stopper = EarlyStopping(patience=50, min_delta=1e-4)
    best_val = float("inf")
    best_state = None

    for epoch in range(400):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            pred = model(batch)
            loss = criterion(pred, batch.y.view(-1).to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(DEVICE)
                pred = model(batch)
                vloss = criterion(pred, batch.y.view(-1).to(DEVICE))
                val_losses.append(vloss.item())
        avg_val = float(np.mean(val_losses))

        if avg_val < best_val:
            best_val = avg_val
            best_state = deepcopy(model.state_dict())

        stopper.step(avg_val)
        if stopper.early_stop:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val

N_TRIALS = 200
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS)

best_params = study.best_trial.params
print_section("Best hyperparameters (Optuna)")
print(prepare_display_table(pd.DataFrame([best_params])))

BEST_PARAM_PATH = GCN_DIR / "best_params_optuna.json"
pd.Series(best_params).to_json(BEST_PARAM_PATH)
report_saved_paths([BEST_PARAM_PATH], 'Saved hyperparameter files')


### 4. Final shielding-aware GCN training and evaluation

Purpose: Fit the selected GCN model, export the checkpoint, and generate publication-ready parity plots.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 4. Final shielding-aware GCN training and evaluation
# Purpose: Fit the selected GCN model, export the checkpoint, and generate publication-ready parity plots.
# -----------------------------------------------------------------------------

import os
import numpy as np
import optuna
from copy import deepcopy
import torch
import json

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
GCN_DIR = PROJECT_ROOT / 'results' / 'gcn'
GCN_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = GCN_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)

apply_plot_style()

train_loader = torch.load(GCN_DIR / 'train_loader.pth', weights_only=False)
valid_loader = torch.load(GCN_DIR / 'valid_loader.pth', weights_only=False)
test_loader = torch.load(GCN_DIR / 'test_loader.pth', weights_only=False)

num_node_features = train_loader.dataset[0].x.shape[1]
BEST_PARAM_PATH = GCN_DIR / 'best_params_optuna.json'
if not BEST_PARAM_PATH.exists():
    raise FileNotFoundError(f'Best params file not found: {BEST_PARAM_PATH} (run Cell 18)')

best_params = pd.read_json(BEST_PARAM_PATH, typ='series').to_dict()

model = GCN(
    num_node_features=num_node_features,
    hidden_channels=int(best_params['hidden_channels']),
    num_layers=int(best_params['num_layers']),
    predictor_hidden_feats=int(best_params['predictor_hidden_feats']),
    dropout=float(best_params['dropout']),
    use_residual=True,
    use_global_shielding=True,
    global_feat_dim=1,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(best_params['lr']),
    weight_decay=float(best_params['weight_decay']),
)
criterion = torch.nn.MSELoss()

class EarlyStopping:
    def __init__(self, patience=120, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.best_state = None
        self.early_stop = False

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

stopper = EarlyStopping(patience=150, min_delta=1e-4)

MAX_EPOCHS = 2000
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y.view(-1).to(DEVICE))
        loss.backward()
        optimizer.step()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(DEVICE)
            pred = model(batch)
            vloss = criterion(pred, batch.y.view(-1).to(DEVICE))
            val_losses.append(vloss.item())
    val_loss = float(np.mean(val_losses))

    stopper.step(val_loss, model)

    if epoch % 50 == 0 or epoch == 1:
        print(f'Epoch {epoch:4d} | Val MSE: {val_loss:.6f}')

    if stopper.early_stop:
        print(f'Early stopping at epoch {epoch}. Best Val MSE: {stopper.best_loss:.6f}')
        break

model.load_state_dict(stopper.best_state)

def collect_predictions(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch).detach().cpu().numpy()
            ytrue = batch.y.view(-1).detach().cpu().numpy()
            preds.append(pred)
            trues.append(ytrue)
    return np.concatenate(trues), np.concatenate(preds)

def metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

ytr_train, ypr_train = collect_predictions(train_loader)
ytr_val, ypr_val = collect_predictions(valid_loader)
ytr_test, ypr_test = collect_predictions(test_loader)

train_mae, train_rmse, train_r2 = metrics(ytr_train, ypr_train)
val_mae, val_rmse, val_r2 = metrics(ytr_val, ypr_val)
test_mae, test_rmse, test_r2 = metrics(ytr_test, ypr_test)

print_section('Final GCN Performance (WITH shielding)')
print(metric_row('Train', train_mae, train_rmse, train_r2, label_width=10))
print(metric_row('Validation', val_mae, val_rmse, val_r2, label_width=12))
print(metric_row('Test', test_mae, test_rmse, test_r2, label_width=12))

MODEL_OUT = GCN_DIR / 'gcn_best_with_shield.pt'
torch.save({'model_state_dict': model.state_dict(), 'best_params': best_params}, MODEL_OUT)
report_saved_paths([MODEL_OUT], 'Saved model files')

def parity_plot(y_true, y_pred, title, out_path):
    fig, ax = create_single_axis_figure('tall')
    ax.scatter(y_true, y_pred, s=18, alpha=0.75, edgecolor='none')

    lo, hi = compute_plot_limits(y_true, y_pred, pad_ratio=0.0)
    add_identity_line(ax, lo, hi)
    annotate_panel_text(ax, metric_text(*metrics(y_true, y_pred)))

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    style_axes(ax, equal_aspect=True)

    fig.tight_layout()
    saved_paths = save_figure(fig, out_path)
    plt.show()
    report_saved_paths(saved_paths, 'Saved figure files')

FIG_DIR = GCN_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
parity_plot(ytr_train, ypr_train, 'GCN (WITH shielding): Train parity', FIG_DIR / 'gcn_parity_train.png')
parity_plot(ytr_val, ypr_val, 'GCN (WITH shielding): Validation parity', FIG_DIR / 'gcn_parity_val.png')
parity_plot(ytr_test, ypr_test, 'GCN (WITH shielding): Test parity', FIG_DIR / 'gcn_parity_test.png')

df_summary = pd.DataFrame([{
    'Model': 'GCN_with_shielding',
    'Train_MAE': train_mae, 'Train_RMSE': train_rmse, 'Train_R2': train_r2,
    'Val_MAE': val_mae, 'Val_RMSE': val_rmse, 'Val_R2': val_r2,
    'Test_MAE': test_mae, 'Test_RMSE': test_rmse, 'Test_R2': test_r2,
    'Checkpoint': str(MODEL_OUT),
    'BestParams': str(best_params),
}])

OUT_SUMMARY = TABLE_DIR / 'gcn_with_shielding_summary.csv'
df_summary.to_csv(OUT_SUMMARY, index=False)
report_saved_paths([OUT_SUMMARY], 'Saved summary tables')
display(prepare_display_table(df_summary))


### 5. Shielding-aware GCN workflow for the Cluster 1 subset

Purpose: Repeat the graph-learning workflow for the cleaned Cluster 1 subset and archive the corresponding artifacts.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 5. Shielding-aware GCN workflow for the Cluster 1 subset
# Purpose: Repeat the graph-learning workflow for the cleaned Cluster 1 subset and archive the corresponding artifacts.
# -----------------------------------------------------------------------------

import os
import random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

from rdkit import Chem
from rdkit.Chem import AllChem, EState, rdMolDescriptors

# =========================================================
# 0) Reproducibility
# =========================================================
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

apply_plot_style()

# =========================================================
# 1) Paths & Input CSV
# =========================================================
PROJECT_ROOT = Path(".").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "gcn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# >>> Change this to your actual CSV file name (must be under data/processed by default)
INPUT_CSV = PROCESSED_DIR / "cleaned_shift_data_with_shield_cluster1.csv"
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

required_cols = {"SMILES", "shift_value", "shielding_value"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

# =========================================================
# 2) Stratified Split (Train/Val/Test)
# =========================================================
# Same bins you used (you can keep as-is for consistency)

X_all = df[["SMILES", "shielding_value"]].copy()
y_all = df["shift_value"].astype(float).copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.10, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1/9), random_state=SEED
)

train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([X_val.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)
test_df  = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

print_section("Split Summary")
print(f"Total: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Save split CSVs (optional but useful)
train_df.to_csv(TABLE_DIR / "train_cluster1.csv", index=False)
val_df.to_csv(TABLE_DIR / "valid_cluster1.csv", index=False)
test_df.to_csv(TABLE_DIR / "test_cluster1.csv", index=False)

# =========================================================
# 3) Atom-level feature configuration
# =========================================================
feature_config = {
    "atom_type": {"encode": True, "type": "categorical"},
    "atom_mass": {"encode": True, "type": "continuous"},
    "hybridization": {"encode": True, "type": "categorical"},
    "is_aromatic": {"encode": False, "type": "boolean"},
    "formal_charge": {"encode": True, "type": "continuous"},
    "chiral_tag": {"encode": False, "type": "boolean"},
    "num_hydrogens": {"encode": True, "type": "continuous"},
    "degree": {"encode": True, "type": "continuous"},
    "implicit_valence": {"encode": True, "type": "continuous"},
    "explicit_valence": {"encode": True, "type": "continuous"},
    "gasteiger_charge": {"encode": True, "type": "continuous"},
    "logp_contrib": {"encode": True, "type": "continuous"},
    "mr_contrib": {"encode": True, "type": "continuous"},
    "estate": {"encode": True, "type": "continuous"},
    "radius": {"encode": True, "type": "continuous"},
    "electronegativity": {"encode": True, "type": "continuous"},
    "is_fluorine": {"encode": False, "type": "boolean"},
}

electronegativities = {
    "H": 2.20, "C": 2.55, "N": 3.04, "O": 3.44, "F": 3.98,
    "P": 2.19, "S": 2.58, "Cl": 3.16, "Si": 1.98, "B": 2.04,
    "Br": 2.96, "I": 4.36, "Na": 0.93, "K": 0.82,
}

# =========================================================
# 4) Graph construction utils
# =========================================================
UNK_TOKEN = "__UNK__"

def add_gasteiger_charges(mol):
    AllChem.ComputeGasteigerCharges(mol)
    charges = []
    for atom in mol.GetAtoms():
        try:
            charges.append(float(atom.GetProp("_GasteigerCharge")))
        except Exception:
            charges.append(0.0)
    return charges

def calculate_atomic_descriptors(mol):
    mol_h = Chem.AddHs(mol)
    logp_contribs, mr_contribs = zip(*rdMolDescriptors._CalcCrippenContribs(mol_h))
    estate_indices = EState.EStateIndices(mol_h)
    pt = Chem.GetPeriodicTable()
    radii = [pt.GetRvdw(atom.GetAtomicNum()) for atom in mol_h.GetAtoms()]
    return logp_contribs, mr_contribs, estate_indices, radii

def molecule_to_graph(mol):
    if mol is None:
        return None

    gasteiger_charges = add_gasteiger_charges(mol)
    logp_contribs, mr_contribs, estate_indices, radii = calculate_atomic_descriptors(mol)

    nodes = []
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        nodes.append({
            "atom_type": atom.GetSymbol(),
            "atom_mass": atom.GetMass(),
            "hybridization": str(atom.GetHybridization()),
            "is_aromatic": atom.GetIsAromatic(),
            "formal_charge": atom.GetFormalCharge(),
            "chiral_tag": atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
            "num_hydrogens": atom.GetTotalNumHs(),
            "degree": atom.GetDegree(),
            "implicit_valence": atom.GetImplicitValence(),
            "explicit_valence": atom.GetExplicitValence(),
            "gasteiger_charge": gasteiger_charges[idx],
            "logp_contrib": logp_contribs[idx],
            "mr_contrib": mr_contribs[idx],
            "estate": estate_indices[idx],
            "radius": radii[idx],
            "electronegativity": electronegativities.get(atom.GetSymbol(), 0.0),
            "is_fluorine": 1 if atom.GetSymbol() == "F" else 0,
        })

    edges = []
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        edges.append([a, b])
        edges.append([b, a])

    if len(edges) == 0:
        edges = [[0, 0]]  # self-loop placeholder

    return nodes, edges

def fit_categorical_maps(all_train_nodes):
    cat_maps = {}
    for feat, cfg in feature_config.items():
        if cfg["encode"] and cfg["type"] == "categorical":
            vocab = sorted(set([n[feat] for n in all_train_nodes]))
            if UNK_TOKEN not in vocab:
                vocab = vocab + [UNK_TOKEN]
            cat_maps[feat] = {v: i for i, v in enumerate(vocab)}
    return cat_maps

def fit_scalers(all_train_nodes):
    scalers = {}
    for feat, cfg in feature_config.items():
        if cfg["encode"] and cfg["type"] == "continuous":
            arr = np.array([float(n[feat]) for n in all_train_nodes]).reshape(-1, 1)
            sc = StandardScaler()
            sc.fit(arr)
            scalers[feat] = sc
    return scalers

def transform_nodes(nodes, cat_maps, scalers):
    out = []
    for n in nodes:
        row = []
        for feat, cfg in feature_config.items():
            if cfg["type"] == "categorical" and cfg["encode"]:
                mp = cat_maps[feat]
                row.append(float(mp.get(n[feat], mp[UNK_TOKEN])))
            elif cfg["type"] == "continuous" and cfg["encode"]:
                row.append(float(scalers[feat].transform([[float(n[feat])]])[0][0]))
            elif cfg["type"] == "boolean":
                row.append(float(int(n[feat])))
        out.append(row)
    return np.array(out, dtype=np.float32)

def df_to_pyg_list(data_df, cat_maps, scalers):
    data_list = []
    invalid = 0

    for _, row in data_df.iterrows():
        smiles = str(row["SMILES"]).strip()
        y_val = float(row["shift_value"])
        shielding_val = float(row["shielding_value"])

        mol = Chem.MolFromSmiles(smiles)
        g = molecule_to_graph(mol)
        if g is None:
            invalid += 1
            continue

        nodes, edges = g
        x_np = transform_nodes(nodes, cat_maps, scalers)
        x = torch.tensor(x_np, dtype=torch.float)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        y_tensor = torch.tensor([y_val], dtype=torch.float)

        data = Data(x=x, edge_index=edge_index, y=y_tensor, smiles=smiles)
        data.shielding = torch.tensor([shielding_val], dtype=torch.float)
        data_list.append(data)

    return data_list, invalid

# =========================================================
# 5) Fit encoders/scalers on TRAIN only
# =========================================================
train_nodes_all = []
invalid_train = 0

for _, row in train_df.iterrows():
    mol = Chem.MolFromSmiles(str(row["SMILES"]).strip())
    g = molecule_to_graph(mol)
    if g is None:
        invalid_train += 1
        continue
    nodes, _ = g
    train_nodes_all.extend(nodes)

cat_maps = fit_categorical_maps(train_nodes_all)
scalers = fit_scalers(train_nodes_all)

ENC_SCALER_PATH = RESULTS_DIR / "encoders_scalers_cluster1.pkl"
joblib.dump({"cat_maps": cat_maps, "scalers": scalers, "UNK_TOKEN": UNK_TOKEN}, ENC_SCALER_PATH)

# =========================================================
# 6) Build PyG datasets + loaders
# =========================================================
train_list, inv1 = df_to_pyg_list(train_df, cat_maps, scalers)
val_list,   inv2 = df_to_pyg_list(val_df,   cat_maps, scalers)
test_list,  inv3 = df_to_pyg_list(test_df,  cat_maps, scalers)

print_section("Graph Construction Summary")
print(f"Invalid SMILES removed (train/val/test): {inv1} / {inv2} / {inv3}")
print(f"Train graphs: {len(train_list)} | Val graphs: {len(val_list)} | Test graphs: {len(test_list)}")

BATCH_SIZE = 64
train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(val_list,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_list,  batch_size=BATCH_SIZE, shuffle=False)

# Save loaders (optional)
torch.save(train_loader, RESULTS_DIR / "train_loader_cluster1.pth")
torch.save(valid_loader, RESULTS_DIR / "valid_loader_cluster1.pth")
torch.save(test_loader,  RESULTS_DIR / "test_loader_cluster1.pth")

num_node_features = train_list[0].x.shape[1]
print(f"Node feature dimension (data.x): {num_node_features}")

# =========================================================
# 7) GCN model definition
# =========================================================
class GCN(torch.nn.Module):
    def __init__(
        self,
        num_node_features: int,
        hidden_channels: int = 128,
        num_layers: int = 3,
        predictor_hidden_feats: int = 256,
        dropout: float = 0.2,
        use_residual: bool = True,
        use_global_shielding: bool = True,
        global_feat_dim: int = 1,
    ):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()

        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))

        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None

        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        g = global_mean_pool(x, batch)

        if self.use_global_shielding:
            if not hasattr(data, "shielding"):
                raise AttributeError("Data object missing 'shielding' attribute while use_global_shielding=True")
            shielding = data.shielding.view(-1, 1).to(g.dtype)
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()

# =========================================================
# 8) Best hyperparameters (from your Optuna result)
# =========================================================
best_params = {
    "hidden_channels": 512,
    "num_layers": 3,
    "predictor_hidden_feats": 1024,
    "dropout": 0.08343724004088336,
    "lr": 0.010585393028555216,
    "weight_decay": 1.7618331701694268e-06,
}

model = GCN(
    num_node_features=num_node_features,
    hidden_channels=int(best_params["hidden_channels"]),
    num_layers=int(best_params["num_layers"]),
    predictor_hidden_feats=int(best_params["predictor_hidden_feats"]),
    dropout=float(best_params["dropout"]),
    use_residual=True,
    use_global_shielding=True,
    global_feat_dim=1,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(best_params["lr"]),
    weight_decay=float(best_params["weight_decay"]),
)
criterion = torch.nn.MSELoss()

# =========================================================
# 9) Early stopping
# =========================================================
class EarlyStopping:
    def __init__(self, patience=150, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.best_state = None
        self.early_stop = False

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

stopper = EarlyStopping(patience=150, min_delta=1e-4)

# =========================================================
# 10) Train
# =========================================================
MAX_EPOCHS = 2000

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y.view(-1).to(DEVICE))
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(DEVICE)
            pred = model(batch)
            vloss = criterion(pred, batch.y.view(-1).to(DEVICE))
            val_losses.append(vloss.item())
    val_loss = float(np.mean(val_losses))

    stopper.step(val_loss, model)

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d} | Val MSE: {val_loss:.6f}")

    if stopper.early_stop:
        print(f"Early stopping at epoch {epoch}. Best Val MSE: {stopper.best_loss:.6f}")
        break

# Restore best model
model.load_state_dict(stopper.best_state)

# =========================================================
# 11) Evaluation helpers
# =========================================================
def collect_predictions(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch).detach().cpu().numpy()
            ytrue = batch.y.view(-1).detach().cpu().numpy()
            preds.append(pred)
            trues.append(ytrue)
    return np.concatenate(trues), np.concatenate(preds)

def metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

ytr_train, ypr_train = collect_predictions(train_loader)
ytr_val,   ypr_val   = collect_predictions(valid_loader)
ytr_test,  ypr_test  = collect_predictions(test_loader)

train_mae, train_rmse, train_r2 = metrics(ytr_train, ypr_train)
val_mae,   val_rmse,   val_r2   = metrics(ytr_val, ypr_val)
test_mae,  test_rmse,  test_r2  = metrics(ytr_test, ypr_test)

print_section("Final GCN Performance (WITH shielding)")
print(metric_row('Train', train_mae, train_rmse, train_r2, label_width=12))
print(metric_row('Validation', val_mae, val_rmse, val_r2, label_width=12))
print(metric_row('Test', test_mae, test_rmse, test_r2, label_width=12))

# =========================================================
# 12) Parity plots (publication-style)
# =========================================================
def parity_plot(y_true, y_pred, title, out_path):
    fig, ax = create_single_axis_figure('tall')
    ax.scatter(y_true, y_pred, s=18, alpha=0.75, edgecolor='none')

    lo, hi = compute_plot_limits(y_true, y_pred, pad_ratio=0.0)
    add_identity_line(ax, lo, hi)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)

    annotate_panel_text(ax, metric_text(mae, rmse, r2))

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    style_axes(ax, equal_aspect=True)

    fig.tight_layout()
    saved_paths = save_figure(fig, out_path)
    plt.show()
    report_saved_paths(saved_paths, 'Saved figure files')

parity_plot(ytr_train, ypr_train, "GCN (WITH shielding): Train parity", FIG_DIR / "gcn_parity_train_cluster1.png")
parity_plot(ytr_val,   ypr_val,   "GCN (WITH shielding): Validation parity", FIG_DIR / "gcn_parity_val_cluster1.png")
parity_plot(ytr_test,  ypr_test,  "GCN (WITH shielding): Test parity", FIG_DIR / "gcn_parity_test_cluster1.png")

# =========================================================
# 13) Save model checkpoint + summary
# =========================================================
MODEL_OUT = RESULTS_DIR / "gcn_best_with_shield_cluster1.pt"
torch.save({"model_state_dict": model.state_dict(), "best_params": best_params}, MODEL_OUT)
report_saved_paths([MODEL_OUT], 'Saved model files')

df_summary = pd.DataFrame([{
    "Model": "GCN_with_shielding",
    "Train_MAE": train_mae, "Train_RMSE": train_rmse, "Train_R2": train_r2,
    "Val_MAE": val_mae, "Val_RMSE": val_rmse, "Val_R2": val_r2,
    "Test_MAE": test_mae, "Test_RMSE": test_rmse, "Test_R2": test_r2,
    "Checkpoint": str(MODEL_OUT),
    "BestParams": str(best_params),
    "InputCSV": str(INPUT_CSV),
}])

OUT_SUMMARY = TABLE_DIR / "gcn_with_shielding_summary_cluster1.csv"
df_summary.to_csv(OUT_SUMMARY, index=False)
report_saved_paths([OUT_SUMMARY], 'Saved summary tables')
display(prepare_display_table(df_summary))


### 6. Compare ExtraTrees and GCN on the Test Split

Purpose: Compare the shielding-aware ExtraTrees model and the shielding-aware GCN on the shared test split.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees and GCN on the test split
# Purpose: Load the saved tabular and graph models, evaluate them on their test data, and export the side-by-side parity comparison.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

apply_plot_style()
PROJECT_ROOT = Path('.').resolve()
FEATURE_WITH_SIGMA = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield.parquet'
MODEL_WITH_SIGMA = PROJECT_ROOT / 'results' / 'models' / 'best_on_val_refined_ExtraTrees_with_shield.pkl'
GCN_TEST_LOADER_PTH = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader.pth'
GCN_CKPT_PATH = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield.pt'

OUT_DIR = PROJECT_ROOT / 'results' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / 'Figure5_ExtraTrees_vs_GCN_Test_AB.png'

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def plain_split_tabular(X, y, seed=42):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1 / 9), random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

def load_xy_predict_extratrees(feature_path, model_path, seed=42):
    df = pd.read_parquet(feature_path)
    required_cols = {'SMILES', 'shielding_value', 'shift_value'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Parquet missing required columns: {missing}. Found: {list(df.columns)}')

    X = df.drop(columns=['shift_value'])
    y = df['shift_value'].astype(float)
    X_train, X_val, X_test, y_train, y_val, y_test = plain_split_tabular(X, y, seed=seed)
    X_test_tab = X_test.drop(columns=['SMILES']) if 'SMILES' in X_test.columns else X_test

    model = joblib.load(model_path)
    if isinstance(model, dict) and 'model' in model:
        model = model['model']

    y_pred_test = model.predict(X_test_tab)
    return y_test.values, y_pred_test, regression_metrics(y_test, y_pred_test)

class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels=512, num_layers=3, predictor_hidden_feats=1024, dropout=0.08343724004088336, use_residual=True, use_global_shielding=True, global_feat_dim=1):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))
        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        g = global_mean_pool(x, batch)
        if self.use_global_shielding:
            shielding = data.shielding.view(-1, 1).to(g.dtype)
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()

def load_gcn_from_ckpt(ckpt_path, num_node_features, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    best_params = ckpt.get('best_params', {})
    model = GCN(
        num_node_features=num_node_features,
        hidden_channels=int(best_params.get('hidden_channels', 512)),
        num_layers=int(best_params.get('num_layers', 3)),
        predictor_hidden_feats=int(best_params.get('predictor_hidden_feats', 1024)),
        dropout=float(best_params.get('dropout', 0.08343724004088336)),
        use_residual=True,
        use_global_shielding=True,
        global_feat_dim=1,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_et, y_pred_et, m_et = load_xy_predict_extratrees(FEATURE_WITH_SIGMA, MODEL_WITH_SIGMA, seed=SEED)
print('ExtraTrees + shielding (test split from parquet) metrics:', m_et)

test_loader = torch.load(GCN_TEST_LOADER_PTH, map_location=DEVICE)
first_batch = next(iter(test_loader))
num_node_features = first_batch.x.shape[1]
print('GCN num_node_features (from test_loader):', num_node_features)

gcn = load_gcn_from_ckpt(GCN_CKPT_PATH, num_node_features=num_node_features, device=DEVICE)
trues, preds = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        pred = gcn(batch).detach().cpu().numpy()
        ytrue = batch.y.view(-1).detach().cpu().numpy()
        preds.append(pred)
        trues.append(ytrue)

y_true_gcn = np.concatenate(trues)
y_pred_gcn = np.concatenate(preds)
m_gcn = regression_metrics(y_true_gcn, y_pred_gcn)
print(metric_row('GCN', *m_gcn))

vmin, vmax = compute_plot_limits(y_true_et, y_pred_et, y_true_gcn, y_pred_gcn)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_et, y_pred_et, 'ExtraTrees with shielding (test)', m_et, vmin, vmax)
parity(axes[1], y_true_gcn, y_pred_gcn, 'GCN with shielding (test)', m_gcn, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 7. Compare GCN Performance Before and After Cluster 1 Cleaning

Purpose: Compare GCN performance before and after applying the Cluster 1 cleaning strategy.


In [ ]:
# -----------------------------------------------------------------------------
# Compare GCN performance before and after Cluster 1 cleaning
# Purpose: Evaluate the two saved GCN checkpoints on their respective test splits and export the before-versus-after comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

PROJECT_ROOT = Path('.').resolve()
apply_plot_style()

GCN_TEST_LOADER_BEFORE = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader.pth'
GCN_CKPT_BEFORE = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield.pt'
GCN_TEST_LOADER_AFTER = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader_cluster1.pth'
GCN_CKPT_AFTER = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield_cluster1.pt'

OUT_DIR = PROJECT_ROOT / 'results' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / 'Figure5_GCN_Cleaning_Test_AB.png'

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels=512, num_layers=3, predictor_hidden_feats=1024, dropout=0.08343724004088336, use_residual=True, use_global_shielding=True, global_feat_dim=1):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))
        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        g = global_mean_pool(x, batch)
        if self.use_global_shielding:
            if not hasattr(data, 'shielding'):
                raise AttributeError('Data object missing shielding while use_global_shielding=True')
            shielding = data.shielding.view(-1, 1).to(g.dtype)
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()

def load_gcn_from_ckpt(ckpt_path, num_node_features, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    best_params = ckpt.get('best_params', {})
    model = GCN(
        num_node_features=num_node_features,
        hidden_channels=int(best_params.get('hidden_channels', 512)),
        num_layers=int(best_params.get('num_layers', 3)),
        predictor_hidden_feats=int(best_params.get('predictor_hidden_feats', 1024)),
        dropout=float(best_params.get('dropout', 0.08343724004088336)),
        use_residual=True,
        use_global_shielding=True,
        global_feat_dim=1,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def eval_on_loader(test_loader_pth, ckpt_path):
    loader = torch.load(test_loader_pth, map_location=DEVICE)
    first_batch = next(iter(loader))
    num_node_features = first_batch.x.shape[1]
    model = load_gcn_from_ckpt(ckpt_path, num_node_features=num_node_features, device=DEVICE)

    trues, preds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch).detach().cpu().numpy()
            ytrue = batch.y.view(-1).detach().cpu().numpy()
            preds.append(pred)
            trues.append(ytrue)

    y_true = np.concatenate(trues)
    y_pred = np.concatenate(preds)
    return y_true, y_pred, regression_metrics(y_true, y_pred)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_A, y_pred_A, mA = eval_on_loader(GCN_TEST_LOADER_BEFORE, GCN_CKPT_BEFORE)
print_section('Figure input metrics')
print(metric_row('Before cleaning', *mA))
y_true_B, y_pred_B, mB = eval_on_loader(GCN_TEST_LOADER_AFTER, GCN_CKPT_AFTER)
print(metric_row('After cleaning', *mB))

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_A, y_pred_A, 'GCN with shielding (before cleaning)', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'GCN with shielding (after cleaning)', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')
